# EDA — what the data looks like before any model

Thin by design: every function comes from `src/triage`, so anything worth
keeping is tested and reusable, and nothing here can disagree with the
pipeline. Outputs are stripped on commit (`nbstripout`), so run it to see them.

Needs `data/interim/base.parquet` — run `make data` first.

In [ ]:
from pathlib import Path

import pandas as pd
from hydra import compose, initialize_config_dir

from triage.data.split import deployment_protocol
from triage.features.sentinels import sentinel_report, zero_variance_columns

ROOT = Path.cwd().parent
with initialize_config_dir(version_base="1.3", config_dir=str(ROOT / "configs")):
    cfg = compose(config_name="config", overrides=[f"paths.root={ROOT}"])

frame = pd.read_parquet(ROOT / "data" / "interim" / "base.parquet")
frame.shape

## Fraud rate by month

The only time signal in the data. Volume falls and fraud rises across the eight
months, which is why every split is by month and why the drift monitor has
something to find.

In [ ]:
by_month = frame.groupby("month").agg(
    applications=("fraud_bool", "size"),
    frauds=("fraud_bool", "sum"),
    fraud_rate=("fraud_bool", "mean"),
)
by_month["fraud_rate_pct"] = (by_month["fraud_rate"] * 100).round(3)
by_month[["applications", "frauds", "fraud_rate_pct"]]

## Fraud rate by age band

Fraud is genuinely more common among older applicants here. That is the root of
the fairness problem: a model that ranks well will stop more older applicants,
and dropping the age column does not remove the signal.

In [ ]:
from triage.fairness.metrics import age_band

banded = frame.assign(band=age_band(frame["customer_age"].to_numpy(), 10))
by_band = banded.groupby("band").agg(
    applications=("fraud_bool", "size"),
    fraud_rate=("fraud_bool", "mean"),
)
by_band["fraud_rate_pct"] = (by_band["fraud_rate"] * 100).round(3)
by_band[["applications", "fraud_rate_pct"]].sort_index()

In [ ]:
older = frame["customer_age"] >= cfg.features.protected.age_cut
print(f"share aged 50 or over: {older.mean():.3%}")
print(f"fraud rate, 50 and over: {frame.loc[older, 'fraud_bool'].mean():.3%}")
print(f"fraud rate, under 50:    {frame.loc[~older, 'fraud_bool'].mean():.3%}")

## Missing values arrive as negative numbers

There are no nulls anywhere. Six columns use a negative sentinel; two others are
negative *without* being missing, and treating those as missing would corrupt
them (DECISIONS D10).

In [ ]:
print("nulls anywhere:", int(frame.isna().sum().sum()))
print("zero-variance columns:", zero_variance_columns(frame))
sentinel_report(frame, cfg)

In [ ]:
# credit_risk_score is negative for a real fraction of rows, and those are real
# scores, not missing. This is the check behind DECISIONS D10.
negative = frame["credit_risk_score"] < 0
print(f"credit_risk_score negative: {negative.mean():.2%}")
print(f"  of which exactly -1: {(frame['credit_risk_score'] == -1).sum():,} rows")
print(
    f"velocity_6h negative: {(frame['velocity_6h'] < 0).sum():,} rows "
    f"(min {frame['velocity_6h'].min():.1f})"
)

## The splits

Train on months 0-4, cut month 5 three ways, hold out 6 and 7. The number to
remember is how much fraud ends up in `cal_conf`: every conformal threshold is
an order statistic of that set.

In [ ]:
splits = deployment_protocol(frame, cfg)
for name, size in splits.sizes().items():
    print(f"  {name:14s} {size:>8,}")
print(f"\nfrauds in cal_conf: {int(frame.loc[splits.cal_conf, 'fraud_bool'].sum()):,}")